# Total — full_game: a totals model on the nonlinear insights (issue #10)

**Question:** can the nonlinearities found in `00_data/05_nonlinearity_insights` (shootout
regression to the mean, pace tails, talent-gap tails, low returning production, early
season, game script) be turned into a model that beats the market on totals, and beats
the one-line rule B (under when `exp_total > 63.5`)?

**Setup**
- One row per game (`src/canes_cfb/totals.py`), both teams' as-of features.
- Target: under vs the **closing** total (available 2016–2025, so more training years).
- Walk-forward: each season 2019–2023 predicted by a model trained on earlier seasons;
  2024–2025 predicted by a model trained through 2023.
- Betting is graded against the **opening** total (2021+), the number you can bet.
- ⚠️ 2024–2025 were already seen in the insights notebook, so they're a *partially*
  clean check. The clean test is 2026, on paper.

In [ ]:
import json

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from canes_cfb.betting import to_games
from canes_cfb.modeling import SPECS, fit_predict
from canes_cfb.paths import PROCESSED, RAW, ROOT
from canes_cfb.totals import (
    BASE_FEATURES,
    NONLINEAR_FEATURES,
    SHOOTOUT_EXP_TOTAL,
    add_line_features,
    build_totals_table,
)

features = pd.read_parquet(PROCESSED / "team_games.parquet")
games = pd.read_parquet(RAW / "games.parquet")
table = build_totals_table(features, games)
table = table[table.completed & ~table.shortened & table.total_close.notna()]
table = table[table.season.between(2016, 2025)].reset_index(drop=True)
vs_close = add_line_features(table, "total_close")
vs_open = add_line_features(table, "total_open")  # NaN line before 2021
print(len(table), "games")

## 1. Engineered nonlinear features

| Feature | Definition | Insight behind it |
|---|---|---|
| `shootout` | exp_total > 63.5 | Ratings and market overstate shootouts |
| `exp_total_z2` | ((exp_total − 55)/8)² | Curvature at both extremes (SHAP shape) |
| `line_high` | line > 62.5 | High closing totals lean under |
| `pace_tail` | game_pace > 4.8 | Very fast games lean under |
| `talent_gap_tail` | max(talent gap − 250, 0) | Only big talent gaps matter |
| `ret_low` | min returning production < 0.35 | Roster turnover threshold |
| `early_season` | weeks 1–3 | Early weeks lean under |
| `blowout_script` | max(\|exp margin\| − 14, 0) | Game script: clock running in blowouts |
| `gap_x_shootout` | (exp_total − line) × shootout | Does disagreement mean something different in shootouts? |

All thresholds are fixed constants chosen from 2016–2023.

In [ ]:
def logistic():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(C=0.05, max_iter=2000),
    )


def gbm(**kw):
    params = dict(
        n_estimators=400,
        learning_rate=0.01,
        num_leaves=2,
        min_child_samples=200,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.7,
        verbose=-1,
    )
    return lambda: lgb.LGBMClassifier(**{**params, **kw})


ALL = BASE_FEATURES + NONLINEAR_FEATURES
MODELS = {
    "logistic, base": (logistic, BASE_FEATURES),
    "logistic, base + nonlinear": (logistic, ALL),
    "lgbm additive, base": (gbm(), BASE_FEATURES),
    "lgbm additive, base + nonlinear": (gbm(), ALL),
    "lgbm depth 3, base + nonlinear": (gbm(num_leaves=8, max_depth=3), ALL),
}
preds = {name: pd.Series(np.nan, index=vs_close.index) for name in MODELS}
for season in range(2019, 2026):
    train = vs_close[(vs_close.season >= 2016) & (vs_close.season < min(season, 2024))]
    train = train[train.under.notna()]
    test = vs_close.season == season
    for name, (make, cols) in MODELS.items():
        model = make().fit(train[cols], train.under.astype(int))
        preds[name][test] = model.predict_proba(vs_close.loc[test, cols])[:, 1]
preds["rule B (shootout → under)"] = pd.Series(
    np.where(vs_close.shootout == 1, 0.56, np.nan), index=vs_close.index
)
preds["always under"] = pd.Series(0.52, index=vs_close.index)

## 2. Results

In [ ]:
def bets(p, frame, threshold):
    ok = p.notna() & frame.under.notna() & frame.line.notna()
    take = ok & (np.maximum(p, 1 - p) >= threshold)
    pick_under = p[take] >= 0.5
    win = 100 * (pick_under == (frame.under[take] == 1)).mean()
    clv = np.where(
        pick_under,
        frame.total_open[take] - frame.total_close[take],
        frame.total_close[take] - frame.total_open[take],
    )
    return int(take.sum()), win, np.nanmean(clv)


rows = []
for name, p in preds.items():
    cv = vs_close.season.between(2019, 2023) & vs_close.under.notna() & p.notna()
    rule = name.startswith(("rule", "always"))
    row = {
        "model": name,
        "log loss (close, 2019-23)": np.nan
        if name.startswith("rule")
        else log_loss(vs_close.under[cv], p[cv].clip(0.01, 0.99)),
    }
    for label, seasons in (("2021-23", (2021, 2023)), ("2024-25", (2024, 2025))):
        m = vs_open.season.between(*seasons)
        n, win, clv = bets(p[m], vs_open[m], 0.5 if rule else 0.53)
        row |= {f"{label} bets": n, f"{label} win % (open)": win, f"{label} CLV": clv}
    rows.append(row)
results = pd.DataFrame(rows).set_index("model")
print("coin-flip log loss: 0.6931 | break-even: 52.38%")
results.round(3)

**Reading it:**
- **No model beats a coin flip against the closing total** (log loss ≈ 0.693). The close
  is very efficient.
- **The nonlinear features don't help.** The simple logistic model looks good in
  2021–23 (≈55%) but falls to ≈50% in 2024–25. LightGBM falls further. That's overfitting.
- **Rule B, one line, is the best and most stable:** ≈56.6% (2021–23) and ≈56.4% (2024–25).
- **"Always under" has positive CLV** (+0.4 to +0.5). Totals drift *down* from open to
  close on average, so part of any under's CLV is just that drift. Rule B's CLV is at or
  below that average: the market doesn't move toward B's picks. Either B is a persistent
  inefficiency or it's luck; only live results will tell.

## 3. The A-vs-B conflict: who's right in shootouts?

In [ ]:
params = json.loads((ROOT / "models" / "team_points_params.json").read_text())
rf = params["random_forest"]["params"]
base = features[features.completed & ~features.shortened & features.season.between(2016, 2025)]
parts = []
for season in range(2021, 2026):  # out-of-sample points predictions, final model recipe
    train, test = base[base.season < season], base[base.season == season].copy()
    test["pred"] = fit_predict(SPECS["random_forest"], rf, train, test)
    parts.append(test)
oos = pd.concat(parts)
keep = [
    "game_id",
    "team_id",
    "pred",
    "season",
    "spread_open",
    "spread_close",
    "total_open",
    "total_close",
    "exp_total",
]
g = to_games(oos[keep], games)
g = g[g.total_open.notna()].assign(edge=lambda d: d.pred_total - d.total_open)
g["result"] = np.sign(g.total - g.total_open)
g = g[g.result != 0]
rule_a = g[g.edge.abs() >= 4]
shoot = rule_a.exp_total > SHOOTOUT_EXP_TOTAL


def rate(d):
    return f"{100 * (np.sign(d.edge) == d.result).mean():.1f}% of {len(d)}"


conflict = g[(g.exp_total > SHOOTOUT_EXP_TOTAL) & (g.edge >= 4)]
pd.Series(
    {
        "rule A, all (2021-25)": rate(rule_a),
        "rule A outside shootouts": rate(rule_a[~shoot]),
        "rule A in shootouts, over picks": rate(rule_a[shoot & (rule_a.edge > 0)]),
        "rule A in shootouts, under picks": rate(rule_a[shoot & (rule_a.edge < 0)]),
        "conflicts (A over vs B under): under won": (
            f"{100 * (conflict.result < 0).mean():.1f}% of {len(conflict)}"
        ),
        **{f"rule A, {s}": rate(d) for s, d in rule_a.groupby("season")},
    }
)

- Rule A's **overs in shootouts lose** (≈48.5%); its unders there win (≈57.6%), which
  agrees with rule B.
- **In the conflict games, the under won only ≈51.5%.** Neither side is worth betting.
- Rule A overall: ≈53.3% over 2021–2025, but with a rising pattern (49.5% in 2021 →
  55–58% in 2024–25).

## 4. Verdict (2026-09-23)

1. **The totals model is not adopted.** Nonlinear features built from the insights don't
   generalize. The insight is real, but a flexible model overfits it, while the simple
   threshold (rule B) captures it.
2. **Conflicts are skipped.** Rules A and B stay graded exactly as registered. The weekly
   summary adds a *portfolio* line (A + B, no bet when they disagree).
3. The honest test for everything is the 2026 season on paper.